In [ ]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

In [205]:
from typing import Annotated, Sequence, TypedDict
from pydantic import BaseModel
from langgraph.graph.message import add_messages
from langchain_core.messages import (
    BaseMessage,
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

class AgentState(TypedDict):
    """Состояние агента, содержащее историю сообщений"""
    messages: Annotated[Sequence[BaseMessage], add_messages]


class Quote(BaseModel):
    quote: str
    author: str

In [206]:
import requests


# def get_quote():
#     response = requests.get(
#         "https://api.forismatic.com/api/1.0/",
#         params={
#             "method": "getQuote",
#             "format": "json",
#             "lang": "ru"
#         }
#     )
#     return response.json()

In [207]:
import aiohttp

from langchain_core.tools import tool


@tool
async def get_quote() -> Quote:
    """Получить случайную мотивационную цитату в формате {"quote": "цитата", "author": "автор"}."""
    try:
        async with aiohttp.ClientSession() as session:
            params = {
                "method": "getQuote",
                "format": "json",
                "lang": "ru"
            }

            async with session.get(
                "https://api.forismatic.com/api/1.0/",
                params=params,
                timeout=aiohttp.ClientTimeout(total=5)
            ) as response:
                # Пробуем декодировать как JSON
                data = await response.json()
                # print(data)
                quote = data.get("quoteText", "").strip()
                author = data.get("quoteAuthor", "").strip()

                if quote:
                    return {"quote": quote, "author": author}
                else:
                    return {"quote": "Работа не волк. Никто не волк. Только волк — волк.",
                            "author": "Джейсон Стетхем"}
    except Exception as e:
        print(f"Ошибка при получении цитаты: {e}")
        return {"quote": "Если закрыть глаза, становится темно.", "author": "Джейсон Стетхем"}

      
tools = [get_quote]

In [208]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent


load_dotenv()

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="alibaba/tongyi-deepresearch-30b-a3b:free",
    temperature=0.3
).bind_tools(tools)

# agent = create_agent(llm, tools)

In [209]:
async def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(
        content="Ты моя система. Ответь на мой вопрос исходя из доступных для тебя инструментов"
    )
    messages = [system_prompt] + list(state["messages"])
    response = await llm.ainvoke(messages)
    return {"messages": [response]}

In [210]:
async def should_continue(state: AgentState) -> str:
    """Проверяет, нужно ли продолжить выполнение или закончить."""
    messages = state["messages"]
    last_message = messages[-1]

    # Если последнее сообщение от AI и содержит вызовы инструментов - продолжаем
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "continue"

    # Иначе заканчиваем
    return "end"

In [212]:
import asyncio
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END


graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

# Настройка потока
graph.add_edge(START, "our_agent")
graph.add_conditional_edges(
    "our_agent", should_continue, {"continue": "tools", "end": END}
)
graph.add_edge("tools", "our_agent")

# Компиляция и запуск
app = graph.compile()


async def main():
    result = await app.ainvoke(
        {
            "messages": [
                HumanMessage(
                    content="Дай мне мотивационную цитату и расскажи пару слов о авторе, если он известен"
                )
            ]
        }
    )

    # Показываем результат
    print("=== Полная история сообщений ===")
    for i, msg in enumerate(result["messages"]):
        print(f"{i+1}. {type(msg).__name__}: {getattr(msg, 'content', None)}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print(f"   Tool calls: {msg.tool_calls}")



await main()

=== Полная история сообщений ===
1. HumanMessage: Дай мне мотивационную цитату и расскажи пару слов о авторе, если он известен
2. AIMessage: 


   Tool calls: [{'name': 'get_quote', 'args': {}, 'id': 'call_aaf00f47d8314ba69e64658f', 'type': 'tool_call'}]
3. ToolMessage: {"quote": "Пусть смотрит человек не на ошибки других, на сделанное и не сделанное другими, но на сделанное и не сделанное им самим.", "author": "Будда Гаутама"}
4. AIMessage: 

Вот мотивационная цитата:  
**"Пусть смотрит человек не на ошибки других, на сделанное и не сделанное другими, но на сделанное и не сделанное им самим."**  
— **Будда Гаутама** (основатель буддизма).  

Эта мысль подчёркивает важность самокритики и ответственности за свои поступки, призывая не судить других, а фокусироваться на собственном развитии. Учение Будды, возникшее более двух тысячелетий назад, остаётся актуальным, предлагая путь к освобождению от страданий через осознанность и добродетель.
